In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_3.py ──
"""
Shared infrastructure for MLFP02 Exercise 3 — A/B testing & multiple comparisons.

Contains: experiment data loading, group extraction, SRM check, common constants,
and small statistical helpers reused across the four technique files:

    01_bootstrap_power.py    — bootstrap CIs + MDE + power curves
    02_hypothesis_testing.py — two-proportion z-test + effect sizes
    03_multiple_testing.py   — Bonferroni + BH-FDR + FDR simulation
    04_permutation_test.py   — distribution-free alternative

Technique-specific code (the actual corrections, permutation loops, power
formulas) does NOT belong here — each technique file owns its own logic.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
N_BOOTSTRAP: int = 10_000
N_PERMUTATIONS: int = 10_000
RANDOM_SEED: int = 42

OUTPUT_DIR = Path("outputs") / "mlfp02_ex3_ab_testing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore e-commerce A/B test
# ════════════════════════════════════════════════════════════════════════


def load_experiment() -> pl.DataFrame:
    """Load the A/B test fixture used across Exercise 3.

    Columns: user_id, experiment_group, metric_value, pre_metric_value,
             revenue, timestamp, segment, platform, country.

    Derives a binary `converted` flag (metric_value > 0) if missing.
    Groups are binarised into {control, treatment}: anything that isn't
    literally "control" is treated as treatment (variant_c, treatment_a,
    etc.). This keeps the two-group tests simple for M2 pedagogy.
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp02", "experiment_data.parquet")

    if "converted" not in df.columns:
        df = df.with_columns(
            (pl.col("metric_value") > 0).cast(pl.Int8).alias("converted")
        )

    df = df.with_columns(
        pl.when(pl.col("experiment_group") == "control")
        .then(pl.lit("control"))
        .otherwise(pl.lit("treatment"))
        .alias("group")
    )
    return df


def split_groups(df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control_df, treatment_df)."""
    control = df.filter(pl.col("group") == "control")
    treatment = df.filter(pl.col("group") == "treatment")
    return control, treatment


def conversion_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_converted, treatment_converted) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["converted"].to_numpy().astype(np.float64)
    t = treatment["converted"].to_numpy().astype(np.float64)
    return c, t


def revenue_arrays(df: pl.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Return (control_revenue, treatment_revenue) as float64 arrays."""
    control, treatment = split_groups(df)
    c = control["revenue"].to_numpy().astype(np.float64)
    t = treatment["revenue"].to_numpy().astype(np.float64)
    return c, t


# ════════════════════════════════════════════════════════════════════════
# SANITY CHECKS — SRM
# ════════════════════════════════════════════════════════════════════════


def srm_check(
    n_control: int, n_treatment: int, expected_ratio: float = 0.5
) -> dict[str, Any]:
    """χ² goodness-of-fit test for Sample Ratio Mismatch.

    Returns dict with chi2, p_value, and a plain-language verdict.
    SRM indicates randomisation bugs, bot traffic, or pipeline issues —
    if p < 0.01 do NOT trust downstream test results.
    """
    n_total = n_control + n_treatment
    expected = np.array([n_total * expected_ratio, n_total * (1 - expected_ratio)])
    observed = np.array([n_control, n_treatment])
    chi2, p = stats.chisquare(observed, f_exp=expected)
    verdict = (
        "SRM DETECTED — investigate randomisation"
        if p < 0.01
        else "OK — sample split consistent"
    )
    return {"chi2": float(chi2), "p_value": float(p), "verdict": verdict}


# ════════════════════════════════════════════════════════════════════════
# SMALL REUSABLE STATS
# ════════════════════════════════════════════════════════════════════════


def two_proportion_ztest(
    p_control: float, p_treatment: float, n_control: int, n_treatment: int
) -> tuple[float, float]:
    """Pooled two-proportion z-test. Returns (z_stat, two_sided_p_value)."""
    p_pool = (p_control * n_control + p_treatment * n_treatment) / (
        n_control + n_treatment
    )
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
    z = (p_treatment - p_control) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p_value)


def cohens_h(p1: float, p2: float) -> float:
    """Cohen's h effect size for two proportions."""
    return float(2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1))))


def cohens_d(x1: np.ndarray, x2: np.ndarray) -> float:
    """Pooled Cohen's d effect size for two samples."""
    s_pool = np.sqrt((x1.var(ddof=1) + x2.var(ddof=1)) / 2)
    return float((x2.mean() - x1.mean()) / s_pool) if s_pool > 0 else 0.0


def interpret_magnitude(abs_effect: float) -> str:
    """Cohen convention: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_effect < 0.2:
        return "negligible"
    if abs_effect < 0.5:
        return "small"
    if abs_effect < 0.8:
        return "medium"
    return "large"


def print_header(title: str) -> None:
    """Consistent banner for each technique file."""
    print("=" * 70)
    print(f"  {title}")
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 3.4: Permutation Test
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement a permutation test from scratch
#   - Understand why permutation tests require no distributional assumptions
#   - Compare permutation p-values with parametric p-values
#   - Apply permutation tests to both conversion and revenue metrics
#   - Visualise the permutation null distribution
#
# PREREQUISITES: Exercise 3.2 (hypothesis testing)
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Load data and compute observed differences
#   2. Permutation test for conversion rate (10K permutations)
#   3. Permutation test for revenue (non-Normal metric)
#   4. Compare with parametric results
#   5. Visualise permutation null distributions
#
# THEORY:
#   Permutation test algorithm:
#     1. Pool ALL observations (control + treatment)
#     2. Randomly assign to "control" and "treatment" (preserving sizes)
#     3. Compute test statistic on permuted data
#     4. Repeat 10K times -> null distribution
#     5. p-value = proportion of |permuted stat| >= |observed stat|
#
#   Key advantage: NO assumptions about the shape of the distribution.
#   If the parametric test assumes Normality and the data is skewed,
#   the permutation test gives a more trustworthy p-value.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats


print_header("MLFP02 Exercise 3.4: Permutation Test")



## TASK 1 — Load data and compute observed differences


In [ ]:
df = load_experiment()
control, treatment = split_groups(df)
n_control = control.height
n_treatment = treatment.height

ctrl_conv, treat_conv = conversion_arrays(df)
ctrl_rev, treat_rev = revenue_arrays(df)

observed_conv_diff = treat_conv.mean() - ctrl_conv.mean()
observed_rev_diff = treat_rev.mean() - ctrl_rev.mean()

print(f"Data loaded: {df.height:,} users")
print(f"  Observed conversion diff: {observed_conv_diff:+.6f}")
print(f"  Observed revenue diff:    ${observed_rev_diff:+.2f}")



### Checkpoint 1


In [ ]:
assert n_control > 0 and n_treatment > 0, "Both groups must have data"
print("\n>>> Checkpoint 1 passed -- data loaded\n")



## THEORY — Why permutation tests?


In [ ]:
# The z-test assumes the sampling distribution of the difference is
# Normal. For conversion rates with large n, the CLT guarantees this.
# But for revenue (highly skewed, heavy-tailed), the CLT approximation
# may be poor even at n=5000.
#
# The permutation test asks a simpler question: "If the treatment had
# NO effect, how likely is it that we'd see a difference this large
# just by random assignment?"
#
# We answer this by ACTUALLY doing the random assignment 10,000 times:
#   - Pool all users together (erasing group labels)
#   - Randomly assign n_control to "control", rest to "treatment"
#   - Compute the difference in the permuted groups
#   - Count how often the permuted difference is as extreme as observed
#
# Analogy: Imagine you're told a coin is biased. You flip it 100 times
# and get 58 heads. Is that enough evidence? You could compute the
# binomial test (parametric). OR you could shuffle a deck of 100 cards
# (58 red, 42 black), deal them into two piles, and count how often
# the difference is as extreme as 58 vs 42. No math needed -- just
# repeated shuffling.



## TASK 2 — Permutation test for conversion rate


In [ ]:
rng = np.random.default_rng(seed=RANDOM_SEED)
all_converted = df["converted"].to_numpy()

perm_conv_diffs = np.zeros(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    # TODO: Randomly permute all_converted using rng.permutation.
    # Then split into first n_control (permuted control) and rest (permuted treatment).
    # Compute the difference: perm_treat_rate - perm_ctrl_rate.
    # Hint: perm = rng.permutation(all_converted)
    #        perm_ctrl_rate = perm[:n_control].mean()
    #        perm_treat_rate = perm[n_control:].mean()
    perm = ____
    perm_ctrl_rate = ____
    perm_treat_rate = ____
    perm_conv_diffs[i] = perm_treat_rate - perm_ctrl_rate

# TODO: Compute permutation p-value = proportion of |perm_diffs| >= |observed|.
# Hint: np.mean(np.abs(perm_conv_diffs) >= np.abs(observed_conv_diff))
perm_p_conversion = ____

print(f"=== Permutation Test (conversion rate) ===")
print(f"Observed difference: {observed_conv_diff:+.6f}")
print(
    f"Permutation null: mean={perm_conv_diffs.mean():.6f}, "
    f"std={perm_conv_diffs.std():.6f}"
)
print(f"Permutation p-value: {perm_p_conversion:.6f}")

# Compare with parametric
_, parametric_p = two_proportion_ztest(
    ctrl_conv.mean(), treat_conv.mean(), n_control, n_treatment
)
print(f"Parametric p-value:  {parametric_p:.6f}")
agreement = (perm_p_conversion < ALPHA) == (parametric_p < ALPHA)
print(f"Agreement: {'YES' if agreement else 'NO'}")



### Checkpoint 2


In [ ]:
assert 0 <= perm_p_conversion <= 1, "Permutation p-value must be valid"
assert len(perm_conv_diffs) == N_PERMUTATIONS, "Should have N_PERMUTATIONS samples"
assert (
    abs(perm_conv_diffs.mean()) < 0.01
), "Permutation null should be centred near zero"
print("\n>>> Checkpoint 2 passed -- conversion permutation test completed\n")



## TASK 3 — Permutation test for revenue (non-Normal)


In [ ]:
# Revenue is typically right-skewed (many small purchases, few large).
# The parametric Mann-Whitney U test is robust to this, but the
# permutation test is even more direct -- no assumptions at all.

all_revenue = df["revenue"].to_numpy().astype(np.float64)

perm_rev_diffs = np.zeros(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    # TODO: Same procedure as above but for revenue.
    # Hint: perm = rng.permutation(all_revenue)
    #        perm_rev_diffs[i] = perm[n_control:].mean() - perm[:n_control].mean()
    perm = ____
    perm_rev_diffs[i] = ____

# TODO: Compute permutation p-value for revenue.
perm_p_revenue = ____

print(f"=== Permutation Test (revenue) ===")
print(f"Observed revenue diff: ${observed_rev_diff:+.2f}")
print(
    f"Permutation null: mean=${perm_rev_diffs.mean():.2f}, "
    f"std=${perm_rev_diffs.std():.2f}"
)
print(f"Permutation p-value: {perm_p_revenue:.6f}")

# Compare with Mann-Whitney U
_, mwu_p = stats.mannwhitneyu(treat_rev, ctrl_rev, alternative="two-sided")
print(f"Mann-Whitney p-value: {mwu_p:.6f}")
rev_agreement = (perm_p_revenue < ALPHA) == (mwu_p < ALPHA)
print(f"Agreement: {'YES' if rev_agreement else 'NO'}")



### Checkpoint 3


In [ ]:
assert 0 <= perm_p_revenue <= 1, "Revenue permutation p-value must be valid"
assert len(perm_rev_diffs) == N_PERMUTATIONS, "Should have N_PERMUTATIONS samples"
print("\n>>> Checkpoint 3 passed -- revenue permutation test completed\n")



## TASK 4 — Comparison summary


In [ ]:
print(f"=== Parametric vs Permutation Comparison ===")
print(f"{'Metric':<20} {'Parametric p':>14} {'Permutation p':>15} {'Agree':>8}")
print("-" * 60)
print(
    f"{'Conversion':<20} {parametric_p:>14.6f} {perm_p_conversion:>15.6f} "
    f"{'YES' if agreement else 'NO':>8}"
)
print(
    f"{'Revenue':<20} {mwu_p:>14.6f} {perm_p_revenue:>15.6f} "
    f"{'YES' if rev_agreement else 'NO':>8}"
)

# INTERPRETATION: When parametric and permutation tests agree, both
# assumptions hold and you can trust either. When they disagree, the
# permutation test is more trustworthy because it makes fewer
# assumptions. In practice, for conversion rates (binary, large n),
# they almost always agree. For revenue (skewed, heavy-tailed),
# disagreement is more common.



### Checkpoint 4


In [ ]:
print("\n>>> Checkpoint 4 passed -- comparison complete\n")



## TASK 5 — Visualise permutation null distributions


In [ ]:
from kailash_ml import ModelVisualizer

viz = ModelVisualizer()

# Conversion permutation null
fig1 = viz.histogram(
    perm_conv_diffs,
    title="Permutation Null: Conversion Rate Difference",
    x_label="Permuted Difference",
)
fig1.add_vline(
    x=observed_conv_diff,
    line_dash="dash",
    annotation_text=f"Observed ({observed_conv_diff:+.4f})",
)
fig1.add_vline(
    x=-abs(observed_conv_diff),
    line_dash="dash",
    line_color="red",
    annotation_text="Mirror",
)
out_1 = OUTPUT_DIR / "permutation_null_conversion.html"
fig1.write_html(str(out_1))
print(f"Saved: {out_1}")

# Revenue permutation null
fig2 = viz.histogram(
    perm_rev_diffs,
    title="Permutation Null: Revenue Difference",
    x_label="Permuted Revenue Difference ($)",
)
fig2.add_vline(
    x=observed_rev_diff,
    line_dash="dash",
    annotation_text=f"Observed (${observed_rev_diff:+.2f})",
)
fig2.add_vline(
    x=-abs(observed_rev_diff),
    line_dash="dash",
    line_color="red",
    annotation_text="Mirror",
)
out_2 = OUTPUT_DIR / "permutation_null_revenue.html"
fig2.write_html(str(out_2))
print(f"Saved: {out_2}")



## APPLY — Distribution-free testing for a Singapore ride-hailing platform


In [ ]:
# SCENARIO: A Singapore ride-hailing platform tests a new surge pricing
# algorithm. The key metric is revenue per ride, which is highly skewed:
# most rides cost S$8-15, but airport rides, peak-hour rides, and
# luxury rides create a heavy right tail (S$50-200).
#
# A t-test on this data is questionable -- the distribution is nowhere
# near Normal, and even with n=10,000 the CLT approximation for the
# mean is unreliable because the variance is dominated by the tail.
#
# The permutation test handles this naturally:
#   - No distributional assumptions
#   - Exact p-value (up to permutation count)
#   - Works for ANY test statistic (mean, median, trimmed mean, ratio)
#
# BUSINESS IMPACT: Without the permutation test, the team would have
# shipped a pricing algorithm that appeared to increase revenue but
# actually just happened to catch a few high-value airport rides in the
# treatment group. Avoiding this false positive saves ~S$200K in
# engineering effort and prevents a price increase that would have
# reduced rider satisfaction with no real revenue benefit.

print(f"\n--- Business Application: Distribution-Free Testing ---")
print("Revenue distributions are rarely Normal.")
print("Permutation tests give reliable p-values regardless of distribution shape.")
print("When parametric and permutation tests disagree, trust the permutation test.")



## REFLECTION


[x] Permutation test: distribution-free, no parametric assumptions
  [x] Permutation null distribution: what "no effect" actually looks like
  [x] Conversion + revenue permutation tests from scratch
  [x] Parametric vs permutation comparison: when they agree and disagree

  EXERCISE 3 COMPLETE: You now have the full A/B testing toolkit:
    3.1: Bootstrap CIs + power analysis (how precise, how sensitive)
    3.2: Hypothesis testing + effect sizes (is it real, is it big enough)
    3.3: Multiple testing corrections (controlling false discoveries)
    3.4: Permutation tests (no assumptions, trustworthy p-values)

  NEXT: In Exercise 4 you'll design a complete experiment from scratch --
  writing hypotheses before data, computing required sample sizes,
  and creating a structured data collection plan.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print(">>> Exercise 3.4 complete -- Permutation Test")

